# 🛒 Online Retail II – Customer Segmentation, Repeat-Purchase Prediction & Spend Forecast

## 🎯 Business Problem
An online gift-ware shop sells to thousands of customers. The owner wants three answers:
1. **Who are my customer groups?** (VIP, regular, low value ...) → *Clustering*
2. **Which customers will buy again in the next 90 days?** → *Classification*
3. **How much will a returning customer spend in the next 90 days?** → *Regression*

With these answers the shop can send offers to the right people instead of everyone.

## 📦 Dataset
**Online Retail II** (UK online store, Dec 2009 – Dec 2011). Columns: `Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country`.

## 🧠 Models used
Logistic Regression, KNN, Gaussian Naive Bayes, Decision Tree, Random Forest, Linear Regression, K-Means, Hierarchical Clustering.

## 🧭 Plan of the notebook
Load → Clean → EDA (chart 1) → Feature engineering → Classification (chart 2) → Regression → Clustering (chart 3) → Conclusion

## 1. Import Important Libraries
pandas + numpy for data work, seaborn + matplotlib for the 3 charts, sklearn for models and scores.

In [1]:
import pandas as pd  # pandas: table (DataFrame) banane aur data saaf karne ke liye
import numpy as np  # numpy: array par tez maths ke liye
import matplotlib.pyplot as plt  # matplotlib: basic charts ke liye
import seaborn as sns  # seaborn: matplotlib ke upar sundar charts ke liye
import warnings  # warnings: bekaar warning messages chhupane ke liye
warnings.filterwarnings("ignore")  # warnings band kar di taaki output saaf dikhe

In [2]:
from sklearn.model_selection import train_test_split  # data ko train aur test hisson mein baantne ke liye
from sklearn.preprocessing import StandardScaler  # saare columns ko same scale par lane ke liye
from sklearn.linear_model import LogisticRegression, LinearRegression  # linear models
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor  # KNN models
from sklearn.naive_bayes import GaussianNB  # Gaussian Naive Bayes classifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor  # decision tree models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor  # random forest models
from sklearn.cluster import KMeans, AgglomerativeClustering  # K-Means aur hierarchical clustering

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix  # classification ke scores
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score  # regression ke error scores
from sklearn.metrics import silhouette_score, adjusted_rand_score  # clustering ki quality ke scores

## 2. Load the Data
Our file is an **Excel (.xlsx)** file with **two sheets** (2009-10 and 2010-11). We read each sheet and join them into one table.
Keep the file in the same folder as this notebook.
⏳ Reading ~1 million rows from Excel takes a few minutes – this is normal.

In [8]:
df_raw = pd.read_csv("online_retail_II.csv")  # csv file padhi
df_raw.shape  # rows aur columns ki ginti

(541910, 8)

In [11]:
df_raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [12]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  str    
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.8 MB


In [13]:
df_raw.describe()

,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## 3. Data Quality Check
Before cleaning we must **see the problems**: missing customers, cancelled invoices, returns and duplicates.

In [14]:
df_raw.isnull().sum()  # har column mein missing values ki ginti

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [ ]:
df_raw.duplicated().sum()

np.int64(5268)

In [ ]:
print(df_raw.duplicated().sum())  # bilkul same (duplicate) rows ki ginti
print((df_raw["Quantity"] <= 0).sum())  # zero ya negative quantity wali rows (returns)
print(df_raw["Invoice"].astype(str).str.startswith("C").sum())  # cancel hue invoices (ye C letter se shuru hote hain)